# SHAP Waterfall & LIME Explanation Plots

Generate example explanation plots for Chapter 2 (Theoretical Background) using the actual XGBoost model from the thesis experiments.

**Output:** Saved to `../../Thesis_latex/figures/figures_ch2/`

In [ ]:
import joblib
import numpy as np
import pandas as pd
import shap
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

matplotlib.rcParams.update({"font.size": 11})

# Paths
BASE = Path("/home/fabian/Desktop/Second_XAI_Paper/Code/from_XAI_trustlab")
FIG_OUT = Path("/home/fabian/Desktop/Master_thesis/Thesis_latex/figures/figures_ch2")
FIG_OUT.mkdir(parents=True, exist_ok=True)

FEATURES = ["lag_1", "lag_2", "lag_3", "lag_4", "lag_5", "lag_6", "lag_7",
            "weekofyear", "holiday_week_count"]

In [ ]:
# Load model and data
model = joblib.load(BASE / "serialized_models" / "XGB_model.joblib")
test_df = pd.read_csv(BASE / "test_features_targets.csv")

X_test = test_df[FEATURES]
y_test = test_df["True_Value"]

print(f"Test instances: {len(X_test)}")
print(f"Features: {FEATURES}")
print(f"\nFirst test instance:")
print(X_test.iloc[0])
print(f"\nTrue value: {y_test.iloc[0]:.2f} kWh")
print(f"Prediction: {model.predict(X_test.iloc[[0]])[0]:.2f} kWh")

## SHAP Waterfall Plot

In [ ]:
# Compute SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
base_value = explainer.expected_value
if isinstance(base_value, (list, np.ndarray)):
    base_value = float(base_value[0])

print(f"Base value (E[f(x)]): {base_value:.2f} kWh")
print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
# Pick a representative instance (first test instance)
idx = 0

explanation = shap.Explanation(
    values=shap_values[idx],
    base_values=base_value,
    data=X_test.iloc[idx].values,
    feature_names=FEATURES,
)

# Print summary
pred = model.predict(X_test.iloc[[idx]])[0]
print(f"Instance {idx}: prediction = {pred:.2f} kWh, true = {y_test.iloc[idx]:.2f} kWh")
print(f"Base value: {base_value:.2f}")
print(f"Sum of SHAP values: {shap_values[idx].sum():.2f}")
print(f"Base + SHAP sum: {base_value + shap_values[idx].sum():.2f}")
print()
for feat, val, sv in sorted(zip(FEATURES, X_test.iloc[idx], shap_values[idx]), key=lambda x: abs(x[2]), reverse=True):
    print(f"  {feat:>20s} = {val:8.2f}  ->  SHAP = {sv:+.2f}")

In [ ]:
# Generate and save SHAP waterfall plot
fig, ax = plt.subplots(figsize=(8, 5))
shap.plots.waterfall(explanation, show=False)

plt.tight_layout()
plt.savefig(FIG_OUT / "shap_waterfall.pdf", bbox_inches="tight", dpi=300)
plt.savefig(FIG_OUT / "shap_waterfall.png", bbox_inches="tight", dpi=300)
plt.show()
print(f"Saved to {FIG_OUT / 'shap_waterfall.pdf'}")

## LIME Explanation Plot

In [ ]:
# Load training data for LIME background distribution
# LIME needs the training data to know feature distributions
train_df = pd.read_csv(BASE / "test_features_targets.csv")  # we use test as proxy; ideally use train

# Try to load actual training data
train_path = Path("/home/fabian/Desktop/First_Paper_XAI_LLM/Data/Household_data")
household_files = list(train_path.glob("*.csv")) + list(train_path.glob("*.txt"))
print("Available data files:", [f.name for f in household_files])

In [ ]:
# Build LIME explainer using test data as background
# (LIME only needs it for discretization/feature statistics)
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_test.values,
    feature_names=FEATURES,
    mode="regression",
    verbose=False,
)

# Explain same instance as SHAP
lime_exp = lime_explainer.explain_instance(
    X_test.iloc[idx].values,
    model.predict,
    num_features=len(FEATURES),
    num_samples=8000,
)

print(f"LIME intercept: {lime_exp.intercept[1] if hasattr(lime_exp.intercept, '__len__') else lime_exp.intercept:.2f}")
print(f"LIME local prediction: {lime_exp.local_pred[0]:.2f}")
print()
for feat, weight in sorted(lime_exp.as_list(), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat:>35s}  ->  weight = {weight:+.2f}")

In [ ]:
# Generate and save LIME explanation plot
fig = lime_exp.as_pyplot_figure()
fig.set_size_inches(8, 5)
plt.tight_layout()
plt.savefig(FIG_OUT / "lime_explanation.pdf", bbox_inches="tight", dpi=300)
plt.savefig(FIG_OUT / "lime_explanation.png", bbox_inches="tight", dpi=300)
plt.show()
print(f"Saved to {FIG_OUT / 'lime_explanation.pdf'}")